<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 3 · Statistics, Machine Learning, and Model Evaluation
### Notebook 03 · Model Evaluation

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh


### Why this notebook exists
Chapter 5 is about comparison. The point is not to chase a fancy score, but
to make the evaluation path explicit and repeatable.

This cell builds the supervised-learning baseline by selecting features, keeping the target separate, and preparing a train/test split.


This cell prepares the notebook for local or Colab execution. It finds the book root, clones the companion repo when needed, and makes the working directory predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "3_data"  # book folder to locate locally or after clone
REPO_URL = "https://github.com/yhilpisch/tdscode"  # companion repo with notebooks, data, and code

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = None  # filled once a matching book directory is found
for candidate in [cwd, *cwd.parents]:  # search upward for the book root
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():  # local book root
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():  # repo/book layout
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")  # Colab clone location
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)  # fetch the companion repo once
    BOOK_ROOT = repo_root / BOOK_NAME  # book folder inside the clone

os.chdir(BOOK_ROOT)  # keep relative paths anchored on the book root
if str(BOOK_ROOT) not in sys.path:  # allow src/ imports
    sys.path.insert(0, str(BOOK_ROOT))  # allow src/ imports

code_dir = BOOK_ROOT / "code"  # helper scripts live here
if code_dir.exists() and str(code_dir) not in sys.path:  # make helper scripts importable
    sys.path.insert(0, str(code_dir))

requirements = BOOK_ROOT / "requirements.txt"  # install only when present
if requirements.exists() and "google.colab" in sys.modules:  # keep Colab in sync with the book
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)  # keep Colab in sync with the book

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

module_path = BOOK_ROOT / 'code' / 'baseline_models.py'
spec = importlib.util.spec_from_file_location('baseline_models', module_path)
assert spec is not None and spec.loader is not None
baseline_models = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline_models)

print(f'Loaded {module_path.resolve()}')


This cell continues the worked example for the current section.


In [ ]:
rng = np.random.default_rng(seed=11)
reg_frame = pd.DataFrame(
    {
        'feature': rng.normal(size=200),
    }
)
reg_frame['target'] = 3.0 * reg_frame['feature'] + rng.normal(scale=0.5, size=200)

rmse = baseline_models.baseline_linear(reg_frame, 'target')
print('baseline linear RMSE =', round(rmse, 3))


This cell builds the supervised-learning baseline by selecting features, keeping the target separate, and preparing a train/test split.


In [ ]:
X, y = make_classification(
    n_samples=300,
    n_features=6,
    n_informative=3,
    n_redundant=1,
    weights=[0.7, 0.3],
    random_state=7,
)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=7,
    stratify=y,
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=7,
    stratify=y_temp,
)

models = {
    'logistic_regression': LogisticRegression(max_iter=1000),
    'decision_tree': DecisionTreeClassifier(max_depth=4, random_state=7),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    for split_name, X_split, y_split in [
        ('validation', X_valid, y_valid),
        ('test', X_test, y_test),
    ]:
        pred = model.predict(X_split)
        rows.append(
            {
                'model': name,
                'split': split_name,
                'accuracy': accuracy_score(y_split, pred),
                'precision': precision_score(y_split, pred, zero_division=0),
                'recall': recall_score(y_split, pred, zero_division=0),
            }
        )

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))


This cell continues the worked example for the current section.


In [ ]:
tree = models['decision_tree']
preds = tree.predict(X_test)
print(confusion_matrix(y_test, preds))


### Where We Are Heading Next
Chapter 6 shifts from evaluation to the feature choices that shape model
behavior before any metric is calculated.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
